---
<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=300>

# **<big><big>Algoritmos y Programación III</big></big>**
**Proyecto final — Clasificación de calidad de frutas**

## Notebook 02 — Limpieza, preprocesamiento, partición y estimación de tamaño

Este cuaderno parte del archivo `data/annotations/labels.csv` generado en el Notebook 01. Su objetivo es dejar los datos listos para los modelos tradicionales y para la CNN.

## 1. Importación de librerías

Importamos las librerías necesarias para abrir imágenes, transformar datos, dividir conjuntos y guardar resultados.

In [6]:
from pathlib import Path
import math
import shutil
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 180)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

## 2. Configuración de rutas y parámetros

Se usan las rutas conocidas del árbol del proyecto.

In [7]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "fruit_quality"
ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "annotations"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

LABELS_PATH = ANNOTATIONS_DIR / "labels.csv"
LABELS_PROCESSED_PATH = ANNOTATIONS_DIR / "labels_processed.csv"
TRAIN_PATH = ANNOTATIONS_DIR / "train.csv"
VAL_PATH = ANNOTATIONS_DIR / "val.csv"
TEST_PATH = ANNOTATIONS_DIR / "test.csv"
PREPROCESSING_SUMMARY_PATH = ANNOTATIONS_DIR / "preprocessing_summary.csv"
INVALID_IMAGES_PATH = ANNOTATIONS_DIR / "invalid_images_after_validation.csv"
REMOVED_DUPLICATES_PATH = ANNOTATIONS_DIR / "removed_duplicate_images_after_eda.csv"

IMAGE_SIZE = 128
TEST_SIZE = 0.20
VAL_SIZE = 0.20
RANDOM_STATE = 42
RESET_PROCESSED_DIR = True
REMOVE_DUPLICATES = True

VALID_QUALITY_LABELS = ["bad", "regular", "good"]
VALID_SIZE_LABELS = ["small", "medium", "large"]

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("LABELS_PATH:", LABELS_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("IMAGE_SIZE:", IMAGE_SIZE)

PROJECT_ROOT: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3
RAW_DIR: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3\data\raw\fruit_quality
LABELS_PATH: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3\data\annotations\labels.csv
PROCESSED_DIR: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3\data\processed
FIGURES_DIR: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3\outputs\figures
IMAGE_SIZE: 128


## 3. Carga del archivo generado por el EDA

El Notebook 01 ya registró las imágenes y creó `labels.csv`. Aquí solo cargamos ese archivo y verificamos que tenga las columnas necesarias para procesar imágenes y particionar los datos.

In [8]:
df = pd.read_csv(LABELS_PATH)

required_columns = [
    "image_id",
    "source",
    "quality_label",
    "fruit_type",
    "relative_path",
    "file_name",
    "width",
    "height"
]

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias en labels.csv: {missing_columns}")

if len(df) == 0:
    raise ValueError("labels.csv está vacío. Agrega imágenes al dataset y ejecuta nuevamente el Notebook 01.")

df = df.drop_duplicates(subset=["image_id"]).reset_index(drop=True)

display(df.head())
print("Filas cargadas:", len(df))
print("Columnas disponibles:", list(df.columns))

ValueError: labels.csv está vacío. Agrega imágenes al dataset y ejecuta nuevamente el Notebook 01.

## 4. Normalización mínima de etiquetas

No volvemos a crear etiquetas, solo limpiamos espacios y mayúsculas para evitar errores en carpetas, particiones y modelos.

In [9]:
df["source"] = df["source"].astype(str).str.lower().str.strip()
df["quality_label"] = df["quality_label"].astype(str).str.lower().str.strip()
df["fruit_type"] = df["fruit_type"].astype(str).str.lower().str.strip()
df["relative_path"] = df["relative_path"].astype(str).str.replace("\\", "/", regex=False).str.strip()

invalid_quality_df = df[~df["quality_label"].isin(VALID_QUALITY_LABELS)].copy()
df = df[df["quality_label"].isin(VALID_QUALITY_LABELS)].copy().reset_index(drop=True)

missing_required_classes = sorted(set(VALID_QUALITY_LABELS) - set(df["quality_label"].unique()))

print("Imágenes con calidad inválida retiradas:", len(invalid_quality_df))
print("Clases de calidad presentes:", sorted(df["quality_label"].unique()))

if missing_required_classes:
    print("Clases de calidad faltantes para la entrega final:", missing_required_classes)

Imágenes con calidad inválida retiradas: 0
Clases de calidad presentes: []
Clases de calidad faltantes para la entrega final: ['bad', 'good', 'regular']


## 5. Validación directa de rutas e imágenes

Nos aseguramos de que las imágenes todavía existen y pueden abrirse antes de redimensionarlas y guardarlas en `data/processed`.

In [10]:
def build_image_path(relative_path):
    return PROJECT_ROOT / Path(relative_path)


def validate_image(path):
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image)
            width, height = image.size
            mode = image.mode

        return True, width, height, mode, ""

    except Exception as error:
        return False, np.nan, np.nan, "", str(error)

resolved_paths = []
validation_records = []

for _, row in df.iterrows():
    resolved_path = build_image_path(row["relative_path"])
    is_valid, width, height, mode, error = validate_image(resolved_path)

    resolved_paths.append(str(resolved_path))
    validation_records.append({
        "is_valid_image": is_valid,
        "validated_width": width,
        "validated_height": height,
        "validated_mode": mode,
        "validation_error": error
    })

validation_df = pd.DataFrame(validation_records)
df["resolved_image_path"] = resolved_paths
df = pd.concat([df, validation_df], axis=1)

invalid_df = df[~df["is_valid_image"]].copy()
df = df[df["is_valid_image"]].copy().reset_index(drop=True)

invalid_df.to_csv(INVALID_IMAGES_PATH, index=False, encoding="utf-8")

print("Imágenes válidas:", len(df))
print("Imágenes inválidas:", len(invalid_df))
print("Reporte de inválidas:", INVALID_IMAGES_PATH)
display(invalid_df.head())

KeyError: 'is_valid_image'

## 6. Retiro de duplicados exactos antes de particionar

El EDA solo reporta duplicados. En este cuaderno se toma la decisión operativa de retirarlos antes de crear `train`, `val` y `test`, para evitar que una misma imagen quede en particiones distintas.

In [11]:
if REMOVE_DUPLICATES and "image_hash" in df.columns:
    duplicated_mask = df.duplicated(subset=["image_hash"], keep="first")
    duplicated_df = df[duplicated_mask].copy()
    df = df[~duplicated_mask].copy().reset_index(drop=True)
elif "image_hash" in df.columns:
    duplicated_df = df[df.duplicated(subset=["image_hash"], keep=False)].copy()
else:
    duplicated_df = pd.DataFrame(columns=df.columns)

duplicated_df.to_csv(REMOVED_DUPLICATES_PATH, index=False, encoding="utf-8")

print("Duplicados retirados:", len(duplicated_df) if REMOVE_DUPLICATES else 0)
print("Imágenes restantes:", len(df))
print("Reporte de duplicados:", REMOVED_DUPLICATES_PATH)
display(duplicated_df.head())

Duplicados retirados: 0
Imágenes restantes: 0
Reporte de duplicados: C:\Users\lunam\OneDrive - Universidad Icesi\VII SEMESTRE\APO III (3)\Proyecto Final\proyecto-final-apo-3\data\annotations\removed_duplicate_images_after_eda.csv


,image_id,source,quality_label,fruit_type,relative_path,file_name,width,height,aspect_ratio,is_square,requires_crop,mode,file_size_kb,image_hash,resolved_image_path


## 7. Lectura estándar de imágenes

Esta función abre cada imagen, corrige orientación EXIF cuando aplica y la convierte a RGB. Se reutiliza en la estimación de tamaño y en el guardado final.

In [12]:
def read_rgb_image(path):
    image = Image.open(path)
    image = ImageOps.exif_transpose(image)
    return image.convert("RGB")

## 8. Estimación de tamaño relativo

El proyecto debe entregar una estimación de tamaño, como no tenemos una medida física en centímetros, calculamos un tamaño relativo dentro de la imagen. La estimación usa la proporción aproximada de la fruta frente al fondo.

In [13]:
def estimate_size_features(path, image_size=128):
    image = read_rgb_image(path)
    original_width, original_height = image.size
    working_image = ImageOps.pad(image, (image_size, image_size), color=(255, 255, 255), centering=(0.5, 0.5))
    array = np.asarray(working_image).astype(np.int16)

    top_border = array[0:5, :, :].reshape(-1, 3)
    bottom_border = array[-5:, :, :].reshape(-1, 3)
    left_border = array[:, 0:5, :].reshape(-1, 3)
    right_border = array[:, -5:, :].reshape(-1, 3)
    border_pixels = np.vstack([top_border, bottom_border, left_border, right_border])
    background_color = np.median(border_pixels, axis=0)

    distance = np.linalg.norm(array - background_color, axis=2)
    threshold = max(25, np.percentile(distance, 70))
    mask = distance > threshold

    area_ratio = float(mask.mean())

    if mask.sum() == 0 or area_ratio < 0.01:
        return {
            "area_ratio": np.nan,
            "diameter_norm": np.nan,
            "bbox_width_norm": np.nan,
            "bbox_height_norm": np.nan,
            "original_width": original_width,
            "original_height": original_height,
            "size_estimation_status": "no_object_detected"
        }

    y_indexes, x_indexes = np.where(mask)
    bbox_width_norm = float((x_indexes.max() - x_indexes.min() + 1) / image_size)
    bbox_height_norm = float((y_indexes.max() - y_indexes.min() + 1) / image_size)
    diameter_norm = float(2 * math.sqrt(area_ratio / math.pi))

    return {
        "area_ratio": area_ratio,
        "diameter_norm": diameter_norm,
        "bbox_width_norm": bbox_width_norm,
        "bbox_height_norm": bbox_height_norm,
        "original_width": original_width,
        "original_height": original_height,
        "size_estimation_status": "estimated"
    }

size_records = []

for index, row in df.iterrows():
    features = estimate_size_features(row["resolved_image_path"], image_size=IMAGE_SIZE)
    size_records.append(features)

    if (index + 1) % 100 == 0:
        print("Imágenes con tamaño estimado:", index + 1)

size_features_df = pd.DataFrame(size_records)
df = pd.concat([df, size_features_df], axis=1)

display(df[["image_id", "fruit_type", "quality_label", "area_ratio", "diameter_norm", "bbox_width_norm", "bbox_height_norm", "size_estimation_status"]].head())
display(df["size_estimation_status"].value_counts().rename_axis("size_estimation_status").reset_index(name="count"))

KeyError: "['area_ratio', 'diameter_norm', 'bbox_width_norm', 'bbox_height_norm', 'size_estimation_status'] not in index"

## 9. Construcción de la etiqueta `size_label`

Creamos tres clases de tamaño: `small`, `medium` y `large`. Primeor, intentamos calcular por fruta para no comparar directamente frutas con formas distintas. Cuando una fruta tiene pocos datos, usamos la distribución global disponible.

In [14]:
def label_size_from_quantiles(values):
    valid_values = values.dropna()

    if len(valid_values) < 3 or valid_values.nunique() < 3:
        return pd.Series(["medium" for _ in values], index=values.index)

    q1 = valid_values.quantile(1 / 3)
    q2 = valid_values.quantile(2 / 3)

    labels = []

    for value in values:
        if pd.isna(value):
            labels.append("medium")
        elif value <= q1:
            labels.append("small")
        elif value <= q2:
            labels.append("medium")
        else:
            labels.append("large")

    return pd.Series(labels, index=values.index)

df["size_label"] = ""

for fruit_type, group in df.groupby("fruit_type"):
    valid_diameters = group["diameter_norm"].dropna()

    if len(group) >= 9 and valid_diameters.nunique() >= 3:
        df.loc[group.index, "size_label"] = label_size_from_quantiles(group["diameter_norm"])

missing_size_indexes = df[df["size_label"] == ""].index

if len(missing_size_indexes) > 0:
    df.loc[missing_size_indexes, "size_label"] = label_size_from_quantiles(df.loc[missing_size_indexes, "diameter_norm"])

df["size_label"] = df["size_label"].astype(str).str.lower().str.strip()
df.loc[~df["size_label"].isin(VALID_SIZE_LABELS), "size_label"] = "medium"

size_counts = df["size_label"].value_counts().reindex(VALID_SIZE_LABELS, fill_value=0)

display(size_counts.rename_axis("size_label").reset_index(name="count"))

,size_label,count
0,small,0
1,medium,0
2,large,0


## 10. Revisión mínima antes de particionar

Aquí solo válidamos que el dataset tenga suficientes imágenes y clases para continuar.

In [16]:
df = df[df["quality_label"].isin(VALID_QUALITY_LABELS)].copy()
df = df[df["size_label"].isin(VALID_SIZE_LABELS)].copy()
df = df.reset_index(drop=True)

if len(df) < 5:
    raise ValueError("No hay suficientes imágenes válidas para crear train, validation y test.")

if df["quality_label"].nunique() < 2:
    raise ValueError("Se necesitan al menos dos clases de calidad para crear una partición útil.")

print("Total de imágenes listas para particionar:", len(df))
print("Clases de calidad presentes:", sorted(df["quality_label"].unique()))
print("Frutas presentes:", sorted(df["fruit_type"].unique()))

ValueError: No hay suficientes imágenes válidas para crear train, validation y test.

## 11. Partición `train`, `val` y `test`

Creamos tres subconjuntos separados para entrenamiento, validación y prueba. La división esperada es más o menos 60%, 20% y 20%.

In [17]:
def choose_stratify_column(data):
    combined = data["quality_label"].astype(str) + "_" + data["fruit_type"].astype(str)
    combined_counts = combined.value_counts()

    if len(combined_counts) > 1 and combined_counts.min() >= 2:
        return combined, "quality_label_fruit_type"

    quality = data["quality_label"].astype(str)
    quality_counts = quality.value_counts()

    if len(quality_counts) > 1 and quality_counts.min() >= 2:
        return quality, "quality_label"

    return None, "random"


def safe_train_test_split(data, test_size, random_state):
    stratify_column, strategy = choose_stratify_column(data)

    try:
        first_data, second_data = train_test_split(
            data,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_column
        )
    except ValueError:
        first_data, second_data = train_test_split(
            data,
            test_size=test_size,
            random_state=random_state,
            stratify=None
        )
        strategy = "random_fallback"

    return first_data, second_data, strategy

train_val_df, test_df = pd.DataFrame(), pd.DataFrame()
train_df, val_df = pd.DataFrame(), pd.DataFrame()

train_val_df, test_df, first_strategy = safe_train_test_split(df, TEST_SIZE, RANDOM_STATE)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df, second_strategy = safe_train_test_split(train_val_df, val_ratio, RANDOM_STATE)

df["split"] = ""
df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

split_counts = df["split"].value_counts().reindex(["train", "val", "test"], fill_value=0)

print("Estrategia primera partición:", first_strategy)
print("Estrategia segunda partición:", second_strategy)
display(split_counts.rename_axis("split").reset_index(name="count"))

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 12. Guardado de imágenes procesadas

Las imágenes se guardan cuadradas y redimensionadas en `data/processed`, organizadas por partición y por calidad. Esta estructura queda lista para cargar datos en los notebooks de modelos.

In [18]:
def prepare_processed_directories():
    if RESET_PROCESSED_DIR and PROCESSED_DIR.exists():
        shutil.rmtree(PROCESSED_DIR)

    for split_name in ["train", "val", "test"]:
        for quality_label in VALID_QUALITY_LABELS:
            (PROCESSED_DIR / split_name / quality_label).mkdir(parents=True, exist_ok=True)


def to_project_relative_path(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")


def save_processed_image(row):
    source_path = Path(row["resolved_image_path"])
    output_dir = PROCESSED_DIR / row["split"] / row["quality_label"]
    output_path = output_dir / f"{row['image_id']}.jpg"

    image = read_rgb_image(source_path)
    processed_image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), color=(255, 255, 255), centering=(0.5, 0.5))
    processed_image.save(output_path, "JPEG", quality=95)

    return to_project_relative_path(output_path)

prepare_processed_directories()

processed_paths = []

for index, row in df.iterrows():
    processed_path = save_processed_image(row)
    processed_paths.append(processed_path)

    if (index + 1) % 100 == 0:
        print("Imágenes guardadas:", index + 1)

df["processed_path"] = processed_paths

print("Imágenes procesadas guardadas:", len(df))
display(df[["image_id", "split", "quality_label", "fruit_type", "size_label", "processed_path"]].head())

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\lunam\\OneDrive - Universidad Icesi\\VII SEMESTRE\\APO III (3)\\Proyecto Final\\proyecto-final-apo-3\\data\\processed'

## 13. Guardado de metadatos procesados

Guardamos los archivos CSV que vamos a usar en los siguientes notebooks.

In [19]:
train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
val_df = df[df["split"] == "val"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

df.to_csv(LABELS_PROCESSED_PATH, index=False, encoding="utf-8")
train_df.to_csv(TRAIN_PATH, index=False, encoding="utf-8")
val_df.to_csv(VAL_PATH, index=False, encoding="utf-8")
test_df.to_csv(TEST_PATH, index=False, encoding="utf-8")

print("Archivo procesado:", LABELS_PROCESSED_PATH)
print("Train:", TRAIN_PATH, len(train_df))
print("Validation:", VAL_PATH, len(val_df))
print("Test:", TEST_PATH, len(test_df))

KeyError: 'split'

## 14. Visualización de la distribución final

Estas tablas y gráficas no repiten el EDA. Solo verifican que la partición final no haya dejado una clase vacía en entrenamiento, validación o prueba.

In [ ]:
split_quality_table = pd.crosstab(df["split"], df["quality_label"]).reindex(index=["train", "val", "test"], columns=VALID_QUALITY_LABELS, fill_value=0)
split_size_table = pd.crosstab(df["split"], df["size_label"]).reindex(index=["train", "val", "test"], columns=VALID_SIZE_LABELS, fill_value=0)

split_quality_table.to_csv(ANNOTATIONS_DIR / "split_quality_distribution.csv", encoding="utf-8")
split_size_table.to_csv(ANNOTATIONS_DIR / "split_size_distribution.csv", encoding="utf-8")

display(split_quality_table)
display(split_size_table)

ax = split_quality_table.plot(kind="bar")
ax.set_title("Distribución de calidad por partición")
ax.set_xlabel("Partición")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_split_quality_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

ax = split_size_table.plot(kind="bar")
ax.set_title("Distribución de tamaño por partición")
ax.set_xlabel("Partición")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_split_size_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Muestra de imágenes procesadas

Se revisa visualmente una muestra pequeña para confirmar que el redimensionamiento y las rutas procesadas quedaron correctas.

In [ ]:
def show_processed_grid(data, title, n=9, random_state=42):
    if data.empty:
        print("No hay imágenes para visualizar.")
        return

    sample = data.sample(min(n, len(data)), random_state=random_state)
    cols = 3
    rows = math.ceil(len(sample) / cols)

    plt.figure(figsize=(cols * 4, rows * 4))

    for index, (_, row) in enumerate(sample.iterrows(), start=1):
        image_path = PROJECT_ROOT / row["processed_path"]
        image = Image.open(image_path).convert("RGB")

        plt.subplot(rows, cols, index)
        plt.imshow(image)
        plt.title(f"{row['split']} | {row['quality_label']} | {row['fruit_type']} | {row['size_label']}")
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_processed_grid(df, "Muestra de imágenes procesadas", n=9, random_state=RANDOM_STATE)

## 16. Resumen final para modelado

Este resumen deja evidencia de cuántas imágenes quedaron listas y cuáles son las rutas que deben usar los siguientes notebooks.

In [ ]:
preprocessing_summary = pd.DataFrame({
    "metric": [
        "total_images_after_validation",
        "removed_invalid_images",
        "removed_duplicate_images",
        "train_images",
        "val_images",
        "test_images",
        "image_size",
        "quality_classes",
        "size_classes",
        "fruit_types",
        "labels_processed_path",
        "train_path",
        "val_path",
        "test_path",
        "processed_directory"
    ],
    "value": [
        len(df),
        len(invalid_df),
        len(duplicated_df) if REMOVE_DUPLICATES else 0,
        len(train_df),
        len(val_df),
        len(test_df),
        IMAGE_SIZE,
        ", ".join(VALID_QUALITY_LABELS),
        ", ".join(VALID_SIZE_LABELS),
        ", ".join(sorted(df["fruit_type"].unique())),
        to_project_relative_path(LABELS_PROCESSED_PATH),
        to_project_relative_path(TRAIN_PATH),
        to_project_relative_path(VAL_PATH),
        to_project_relative_path(TEST_PATH),
        to_project_relative_path(PROCESSED_DIR)
    ]
})

preprocessing_summary.to_csv(PREPROCESSING_SUMMARY_PATH, index=False, encoding="utf-8")

display(preprocessing_summary)
print("Resumen guardado en:", PREPROCESSING_SUMMARY_PATH)

## 17. Conclusiones preliminares

Con este cuaderno quedan listas las imágenes procesadas, la estimación de tamaño y las particiones para los siguientes cuadernos del proyecto.

El Notebook 03 puede usar `data/annotations/labels_processed.csv` para modelos tradicionales con características extraídas de imagen.

El Notebook 04 puede usar `data/processed/train`, `data/processed/val` y `data/processed/test` para entrenar la CNN.